In [39]:
import numpy as np
import pandas as pd
import re
import random

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [40]:
df = pd.read_csv("sqliv2.csv",encoding="utf-16")  # غير الاسم لو مختلف

df.head()

,Sentence,Label
0,NaN,1
1,""" or pg_sleep ( __TIME__ ) --",1
2,create user name identified by pass123 tempora...,1
3,%29,1
4,' AND 1 = utl_inaddr.get_host_address ( ( S...,1


In [41]:
def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s\(\)=\'\"\-\_\*\+\,\.]', '', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()

# ناخد بس الهجمات
texts = df[df['Label'] == 1]['Sentence'].apply(clean_text)

texts = texts.dropna()
texts = texts[texts.str.len() > 10]

texts.head()

1                        " or pg_sleep ( __time__ ) --
2    create user name identified by pass123 tempora...
4    ' and 1 = utl_inaddr.get_host_address ( ( sele...
5    select * from users where id = '1' or 1 = 1 un...
6    select * from users where id = 1 or 1" ( union...
Name: Sentence, dtype: object

In [42]:
texts = ["<start> " + t + " <end>" for t in texts]

In [43]:
seq_length = 40

X = []
y = []

for i in range(len(text) - seq_length):
    seq = text[i:i+seq_length]
    label = text[i+seq_length]
    
    X.append([char_to_idx[c] for c in seq])
    y.append(char_to_idx[label])

X = np.array(X)
y = np.array(y)

# reshape + normalize
X = X.reshape((X.shape[0], X.shape[1], 1))
X = X / float(vocab_size)

print(X.shape, y.shape)

(1165843, 40, 1) (1165843,)


In [45]:
model = Sequential()

model.add(LSTM(256, return_sequences=True, input_shape=(seq_length, 1)))
model.add(Dropout(0.2))

model.add(LSTM(256, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(512, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(128, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(64))  # آخر واحدة بس تكون False
model.add(Dropout(0.2))

model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

model.summary()

c:\Users\User.DESKTOP-OQ6NE4T\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_11 (LSTM)                  │ (None, 40, 256)        │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ (None, 40, 256)        │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ (None, 40, 512)        │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 40, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_14 (LSTM)                  │ (None, 40, 128)        │       328,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 40, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_15 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 100)            │         6,500 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,748,516 (10.48 MB)

 Trainable params: 2,748,516 (10.48 MB)

 Non-trainable params: 0 (0.00 B)

In [46]:
model.fit(X, y, epochs=30, batch_size=16)

Epoch 1/30
  207/72866 ━━━━━━━━━━━━━━━━━━━━ 4:14:22 210ms/step - loss: 3.6701

KeyboardInterrupt: 

In [ ]:
def sample_top_k(preds, k=5):
    preds = np.asarray(preds).astype('float64')
    
    top_k_indices = preds.argsort()[-k:]
    top_k_probs = preds[top_k_indices]
    top_k_probs = top_k_probs / np.sum(top_k_probs)
    
    return np.random.choice(top_k_indices, p=top_k_probs)

In [ ]:
def generate_text(model, length=120):
    start_index = random.randint(0, len(text) - seq_length - 1)
    pattern = text[start_index:start_index + seq_length]
    
    generated = pattern
    
    for _ in range(length):
        x = np.array([char_to_idx[c] for c in pattern])
        x = x.reshape((1, seq_length, 1))
        x = x / float(vocab_size)
        
        prediction = model.predict(x, verbose=0)[0]
        
        index = sample_top_k(prediction, k=5)
        result = idx_to_char[index]
        
        generated += result
        pattern = pattern[1:] + result

        # منع التكرار الغبي
        if len(set(generated[-10:])) == 1:
            break
    
    return generated

In [ ]:
for _ in range(5):
    print("----")
    print(generate_text(model))

In [ ]:
for _ in range(5):
    print("----")
    print(generate_text(model, 120))

In [ ]:
def clean_generated(text):
    allowed = "abcdefghijklmnopqrstuvwxyz0123456789 ()=,'-*+._"
    return ''.join([c for c in text if c in allowed])

for _ in range(5):
    print("----")
    print(clean_generated(generate_text(model)))